# Teste e Análise do modelo

In [17]:
# Importação de bibliotecas
import sys
import os
sys.path.append(os.path.abspath(".."))

from src.features.build_model import construir_e_treinar_modelo
from src.features.predict_model import fazer_predicoes, exibir_predicoes
from src.data.processing_data import processing_data
from src.features.feature_engineering import feature_engineering
from src.features.clean_text import limpeza_texto

import pandas as pd

# Pré-Processamento e Machine Learning
from sklearn.model_selection import train_test_split

# Definição do caminho do dataset
dataset_csv_path = '../data/dataset.csv'

# Carregamento do dataset
df = pd.read_csv(dataset_csv_path)

# Limpeza do dataset
df = processing_data(df)

# Aplicação do tratamento de colunas e valores no dataset para treinamento do modelo
df = feature_engineering(df)

df.head()

Processando a limpeza dos dados...
Aplicando features...


,texto_limpo,sentimento_value
0,estou muito feliz com a compra o cadeira gamer...,1
2,nao recomendo a entrega foi lenta e o celular ...,0
3,o monitor e decepcionante o suporte ao cliente...,1
4,e um livro ok pelo prceo que paguei,0
5,nao rceomendo a entrega atrasou muito e o moni...,1


In [18]:
# Definição das variáveis X (entrada) e y saída 
X = df['texto_limpo']
y = df['sentimento_value']

In [19]:
# Divisão dos dados em treino e teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size = 0.25, random_state = 42, stratify = y)

In [20]:
# Construção e Treinamento do modelo
model = construir_e_treinar_modelo(X_treino, y_treino, X_teste, y_teste)

Inicializando a otimização de hiperparâmetros...
Fitting 5 folds for each of 72 candidates, totalling 360 fits
Melhores hiperparâmetros encontrados:
{'logreg__C': 0.1, 'logreg__max_iter': 5000, 'logreg__penalty': 'l1', 'tfidf__max_features': 500, 'tfidf__ngram_range': (1, 1)}
Acurácia do Modelo: 81.15%

Relatório de classificação:
              precision    recall  f1-score   support

    Negativo       0.80      0.81      0.80        58
    Positivo       0.83      0.81      0.82        64

    accuracy                           0.81       122
   macro avg       0.81      0.81      0.81       122
weighted avg       0.81      0.81      0.81       122

Modelo salvo com sucesso em: models/model_feeedback.pkl


c:\Users\ghost\OneDrive - Fundação Salvador Arena\Python Projects\customer-feedback-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\ghost\OneDrive - Fundação Salvador Arena\Python Projects\customer-feedback-ml\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


## Teste com Novos Reviews

In [21]:
# Definição dos novos reviews para teste
novos_reviews = [
    "A bateria do celular não dura nada, péssima compra.",
    "Chegou antes do prazo e o produto é de ótima qualidade! Estou muito feliz.",
    "O serviço de atendimento foi rápido e eficiente.",
    "Não recomendo, veio faltando peças e a cor estava errada.",
    "Serviço muito bom, com certeza realizarei outras vezes!"
]

print("Reviews para classificação:")
for i, review in enumerate(novos_reviews, 1):
    print(f"{i}. {review}\n")

Reviews para classificação:
1. A bateria do celular não dura nada, péssima compra.

2. Chegou antes do prazo e o produto é de ótima qualidade! Estou muito feliz.

3. O serviço de atendimento foi rápido e eficiente.

4. Não recomendo, veio faltando peças e a cor estava errada.

5. Serviço muito bom, com certeza realizarei outras vezes!



In [22]:
# Limpeza e pré-processamento dos novos reviews
novos_reviews_limpos = [limpeza_texto(review) for review in novos_reviews]

print("Reviews após limpeza:")
for i, review_limpo in enumerate(novos_reviews_limpos, 1):
    print(f"{i}. {review_limpo}\n")

Reviews após limpeza:
1. a bateria do celular nao dura nada pessima compra

2. chegou antes do prazo e o produto e de otima qualidade estou muito feliz

3. o servico de atendimento foi rapido e eficiente

4. nao recomendo veio faltando pecas e a cor estava errada

5. servico muito bom com certeza realizarei outras vezes



In [23]:
# Carregando o melhor modelo treinado
modelo_treinado = model['modelo']

# Fazer previsões nos novos reviews
resultado = fazer_predicoes(modelo_treinado, novos_reviews_limpos)

# Exibir as previsões de forma formatada
exibir_predicoes(resultado)


PREVISÕES DO MODELO

[Review 1]
Texto: a bateria do celular nao dura nada pessima compra
Sentimento: Negativo
Confiança: 92.14%
  - Negativo: 92.14%
  - Positivo: 7.86%
--------------------------------------------------------------------------------

[Review 2]
Texto: chegou antes do prazo e o produto e de otima qualidade estou muito feliz
Sentimento: Positivo
Confiança: 77.89%
  - Negativo: 22.11%
  - Positivo: 77.89%
--------------------------------------------------------------------------------

[Review 3]
Texto: o servico de atendimento foi rapido e eficiente
Sentimento: Negativo
Confiança: 50.00%
  - Negativo: 50.00%
  - Positivo: 50.00%
--------------------------------------------------------------------------------

[Review 4]
Texto: nao recomendo veio faltando pecas e a cor estava errada
Sentimento: Negativo
Confiança: 55.73%
  - Negativo: 55.73%
  - Positivo: 44.27%
--------------------------------------------------------------------------------

[Review 5]
Texto: servico mu

In [24]:
# Criar um DataFrame com os resultados para melhor visualização
df_resultados = pd.DataFrame({
    'Review Original': novos_reviews,
    'Review Limpo': novos_reviews_limpos,
    'Sentimento': resultado['sentimentos'],
    'Confiança (%)': [f"{prob*100:.2f}%" for prob in resultado['probabilidades'].max(axis=1)]
})

print("\nResumo dos Resultados:")
print(df_resultados.to_string(index=False))


Resumo dos Resultados:
                                                           Review Original                                                             Review Limpo Sentimento Confiança (%)
                       A bateria do celular não dura nada, péssima compra.                        a bateria do celular nao dura nada pessima compra   Negativo        92.14%
Chegou antes do prazo e o produto é de ótima qualidade! Estou muito feliz. chegou antes do prazo e o produto e de otima qualidade estou muito feliz   Positivo        77.89%
                          O serviço de atendimento foi rápido e eficiente.                          o servico de atendimento foi rapido e eficiente   Negativo        50.00%
                 Não recomendo, veio faltando peças e a cor estava errada.                  nao recomendo veio faltando pecas e a cor estava errada   Negativo        55.73%
                   Serviço muito bom, com certeza realizarei outras vezes!                    servico muito bom